# Notebook 1: Data Engineering & ETL Pipeline

## 1. Business Context & Objective
ITB distributes three product lines (A, B, C) across four sales channels (Ecommerce, Website, Store, Premium). Raw data was collected from disparate transactional systems, producing three systematic quality issues:

* **Duplicate profiles:** Conflicting demographic and educational attributes across customer records.
* **Channel inconsistency:** Store master uses `stores/premium` while order transactions split them into distinct `stores` and `premium` values.
* **Orphan records:** Transactional records referencing customer IDs absent from the customer dimension.

This notebook resolves these anomalies and exports a clean `master_df.csv` as the single source of truth for downstream analytics.

In [8]:
import pandas as pd
import numpy as np
import os

# Configure pandas options for clean notebook output
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Environment successfully initialized with pandas and numpy.")

Environment successfully initialized with pandas and numpy.


## 2. Data Extraction & Consolidation
Raw data comprises three dimension files (`dim_customer`, `dim_product`, `dim_stores`) and five partitioned transaction files (`dim_order_part_1` through `dim_order_part_5`). The code below resolves file paths dynamically, ingests all parts, and concatenates them into a unified orders DataFrame.

In [9]:
# Resolve data path (supports both project-root and notebooks-subfolder execution)
raw_data_path = 'data/raw/' if os.path.exists('data/raw/') else '../data/raw/'

# Load dimension tables
try:
    df_customer_raw = pd.read_csv(os.path.join(raw_data_path, 'dim_customer.csv'))
    df_stores_raw   = pd.read_csv(os.path.join(raw_data_path, 'dim_stores.csv'))
    df_product_raw  = pd.read_csv(os.path.join(raw_data_path, 'dim_product.csv'))
    print("Dimension tables loaded: Customer, Stores, Product.")
except FileNotFoundError as e:
    print(f"[ERROR] {e}")

# Dynamically locate and concatenate partitioned order files
try:
    order_files = sorted([f for f in os.listdir(raw_data_path)
                          if f.startswith('dim_order_part_') and f.endswith('.csv')])
    order_parts = [pd.read_csv(os.path.join(raw_data_path, f)) for f in order_files]
    df_order_raw = pd.concat(order_parts, ignore_index=True)
    print(f"Order files ingested: {len(order_files)} parts \u2192 {df_order_raw.shape[0]:,} rows \u00d7 {df_order_raw.shape[1]} cols.")
except Exception as e:
    print(f"[ERROR] {e}")

Dimension tables loaded: Customer, Stores, Product.
Order files ingested: 5 parts → 2,107,643 rows × 12 cols.


## 3. Initial Data Profiling
Baseline structural audit across all four raw tables — schema, null counts, and cardinality — to quantify data quality issues prior to transformation.

In [10]:
# Profile raw tables to evaluate structural health
datasets = {
    "Raw Customers": df_customer_raw,
    "Consolidated Orders": df_order_raw,
    "Raw Stores": df_stores_raw,
    "Raw Products": df_product_raw
}

for name, df in datasets.items():
    print(f"=== Profiling: {name} ===")
    print(f"Shape: {df.shape}")
    print("\nData Types & Non-Null Counts:")
    print(df.info())
    print("\nMissing Values Count:")
    print(df.isnull().sum())
    print("\nUnique Cardinality:")
    for col in df.columns:
        print(f" - {col}: {df[col].nunique()} unique values")
    print("="*50 + "\n")

=== Profiling: Raw Customers ===
Shape: (849472, 10)

Data Types & Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 849472 entries, 0 to 849471
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   customer_key     849472 non-null  int64 
 1   customer_id      849472 non-null  int64 
 2   name_full        849472 non-null  object
 3   gender           849472 non-null  object
 4   address          445211 non-null  object
 5   region           849472 non-null  object
 6   dob              849472 non-null  object
 7   age              849472 non-null  int64 
 8   education_level  685804 non-null  object
 9   job_title        670136 non-null  object
dtypes: int64(3), object(7)
memory usage: 64.8+ MB
None

Missing Values Count:
customer_key            0
customer_id             0
name_full               0
gender                  0
address            404261
region                  0
dob               

## 4. Data Cleaning, Type Standardization & Schema Alignment

In this phase, we align schemas and standardize data types across all tables:
1. Parse date columns (`order_date` and `dob`) into proper `datetime64[ns]` formats.
2. Standardize key column names (renaming `store_codes` to `store_code` in the Store master).
3. Handle missing crucial keys by removing null rows in the Store dimension.

In [11]:
# Parse date columns to datetime64
df_order_raw['order_date'] = pd.to_datetime(df_order_raw['order_date'], errors='coerce')
df_customer_raw['dob']     = pd.to_datetime(df_customer_raw['dob'],        errors='coerce')

# Align store dimension: rename key column, drop null codes, cast to int
initial_store_count = len(df_stores_raw)
df_stores_raw.rename(columns={'store_codes': 'store_code'}, inplace=True)
df_stores_raw.dropna(subset=['store_code'], inplace=True)
df_stores_raw['store_code'] = df_stores_raw['store_code'].astype(int)

print(f"Date columns parsed. Removed {initial_store_count - len(df_stores_raw)} store records with null keys.")

Date columns parsed. Removed 3 store records with null keys.


## 5. Resolving Channel Inconsistencies & Orphan Records

**Channel disambiguation:** The store master uses `stores/premium` as a single category while orders distinguish `stores` and `premium` explicitly. Stores with multi-channel transaction history are reclassified as `premium`; the remainder default to `stores`.

**Referential integrity:** Transactional records referencing `customer_id`s absent from the customer dimension (e.g., the dummy ID `-43`) are identified and removed.

In [12]:
# --- Channel Disambiguation ---
# Stores appearing in both 'stores' and 'premium' transactions are reclassified as 'premium'
store_channel_counts = df_order_raw.groupby('store_code')['sales_channel'].nunique()
hybrid_stores = store_channel_counts[store_channel_counts > 1].index

df_stores_raw.loc[df_stores_raw['store_code'].isin(hybrid_stores), 'sales_channel'] = 'premium'
df_stores_raw['sales_channel'] = df_stores_raw['sales_channel'].replace({'stores/premium': 'stores'})

print(f"Channel standardization complete. Store master channels: {df_stores_raw['sales_channel'].unique()}")

# --- Referential Integrity: Remove Orphan Records ---
valid_customer_ids = set(df_customer_raw['customer_id'].unique())
orphan_customer_ids = set(df_order_raw['customer_id'].unique()) - valid_customer_ids

initial_order_count = len(df_order_raw)
df_order_cleaned = df_order_raw[~df_order_raw['customer_id'].isin(orphan_customer_ids)].copy()

print(f"Removed {initial_order_count - len(df_order_cleaned):,} orphan records ({len(orphan_customer_ids)} invalid IDs). Clean count: {len(df_order_cleaned):,}")

Channel standardization complete. Store master channels: ['premium' 'stores' 'ecommerce' 'website']
Removed 56,435 orphan records (619 invalid IDs). Clean count: 2,051,208


## 6. Attribute Standardization on Dimensions
Three cleaning operations applied to customer and store dimensions:
1. Consolidate redundant education labels via a standardized mapping dictionary.
2. Impute unresolvable missing categoricals (`education_level`, `job_title`, `address`) with `'Unknown'` — standard data warehousing practice.
3. Apply `.strip().title()` to geographical text columns to eliminate whitespace/casing duplicates.

In [13]:
# 1. Consolidate redundant education labels
education_mapping = {
    "Bachelor's degree": "Bachelor",
    "Master's degree": "Master",
    "High school education": "High school"
}
df_customer_raw['education_level'] = df_customer_raw['education_level'].replace(education_mapping)

# 2. Impute unresolvable missing categoricals with 'Unknown'
for col in ['education_level', 'job_title', 'address']:
    df_customer_raw[col] = df_customer_raw[col].fillna('Unknown')

# 3. Normalize geographical text (trim whitespace, title-case)
df_customer_raw['region']          = df_customer_raw['region'].str.strip().str.title()
df_stores_raw['name_region']       = df_stores_raw['name_region'].str.strip().str.title()
df_stores_raw['name_district']     = df_stores_raw['name_district'].str.strip().str.title()

print("Attribute standardization complete: education labels mapped, nulls imputed, geography normalized.")

Attribute standardization complete: education labels mapped, nulls imputed, geography normalized.


## 7. Final Merge & Export (master_df)
The year 2021 is excluded: it contains only a single day of records, which would introduce severe YoY distortion. The analytical window is restricted to **2022–2024**.

Consolidation sequence: left-join orders ↔ customers on `customer_id` → extract `year` → filter 2021 → left-join store location attributes (`store_region`, `store_district`) on `store_code` → export to `data/processed/master_df.csv`.

In [14]:
# Step 1: Join transactions with customer attributes
df_master = pd.merge(df_order_cleaned, df_customer_raw, on='customer_id', how='left')

# Step 2: Extract year; exclude 2021 (single-day data — severe sparsity bias)
df_master['year'] = df_master['order_date'].dt.year
initial_master_rows = len(df_master)
df_master = df_master[df_master['year'] != 2021].copy()
print(f"Excluded {initial_master_rows - len(df_master):,} records from 2021. Analytical window: 2022\u20132024 ({len(df_master):,} rows).")

# Step 3: Enrich with store location attributes (deduplicated to prevent fan-out)
store_locations = (
    df_stores_raw[['store_code', 'name_region', 'name_district']]
    .drop_duplicates(subset=['store_code'], keep='first')
    .rename(columns={'name_region': 'store_region', 'name_district': 'store_district'})
)
df_master = pd.merge(df_master, store_locations, on='store_code', how='left')

# Step 4: Export to processed directory
parent_dir = os.path.dirname(os.path.normpath(raw_data_path))
processed_dir = os.path.join(parent_dir, 'processed')
os.makedirs(processed_dir, exist_ok=True)
master_file_path = os.path.join(processed_dir, 'master_df.csv')
df_master.to_csv(master_file_path, index=False, encoding='utf-8-sig')

print(f"master_df exported \u2192 '{master_file_path}'")

Excluded 1,945 records from 2021. Analytical window: 2022–2024 (2,049,263 rows).
master_df exported → '..\data\processed\master_df.csv'
